In [1]:
import requests
import base64

In [2]:
# --- IMPORTS ---
import requests                        # sends HTTP requests (like a browser fetching a page)
from bs4 import BeautifulSoup          # parses the HTML we get back so we can search through it
import pandas as pd                    # helps us store data in a table and save to CSV
import time                            # lets us pause between requests so Amazon doesn't block us
from datetime import datetime          # used to create a timestamp for the output filename

# --- HEADERS ---
# When your browser visits Amazon, it sends these "headers" to identify itself.
# Without these, Amazon sees a plain Python script and blocks it.
# We fake a real Chrome browser on Windows.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-IN,en;q=0.9",        # tells Amazon we want English (India)
    "Accept-Encoding": "gzip, deflate, br",      # accepted compression formats
    "Accept": "text/html,application/xhtml+xml", # we want HTML back
    "Referer": "https://www.amazon.in/",         # pretend we came from Amazon's homepage
}

# --- MAIN SCRAPING FUNCTION ---
def scrape_amazon_laptops(pages=3):
    """
    Scrapes laptop listings from Amazon.in.
    pages = how many search result pages to go through (each has ~20 products)
    """
    products = []   # empty list; we'll add one dict per product here

    for page in range(1, pages + 1):   # loop from page 1 to page 3 (or however many)

        # Build the URL for this page
        # amazon.in uses ?k= for keyword and &page= for the page number
        url = f"https://www.amazon.in/s?k=laptops&page={page}"
        print(f"Scraping page {page}: {url}")

        # Send the GET request with our fake browser headers
        response = requests.get(url, headers=HEADERS)

        # Check if we got a successful response (200 = OK)
        if response.status_code != 200:
            print(f"  Failed to fetch page {page} (status {response.status_code})")
            continue   # skip this page and move to the next one

        # Parse the HTML content using BeautifulSoup
        # "html.parser" is Python's built-in HTML parser (no extra install needed)
        soup = BeautifulSoup(response.content, "html.parser")

        # Find all product cards on the page
        # Each laptop listing has data-component-type="s-search-result" in its div
        # .select() uses CSS selectors - like targeting elements in a webpage
        items = soup.select("div[data-component-type='s-search-result']")
        print(f"  Found {len(items)} products on page {page}")

        for item in items:   # loop through each product card

            # --- TITLE ---
            # The product title sits inside: <h2> -> <a> -> <span>
            title_tag = item.select_one("h2 a span")
            title = title_tag.text.strip() if title_tag else "N/A"
            # .text gets the text content, .strip() removes extra spaces/newlines
            # if title_tag is None (not found), we store "N/A"

            # --- PRICE ---
            # Price is in a <span> with class "a-price-whole"
            # It contains just the number (e.g. "54,990") without the ₹ symbol
            price_tag = item.select_one("span.a-price-whole")
            price = price_tag.text.strip().replace(",", "") if price_tag else "N/A"
            # .replace(",", "") removes commas so "54,990" becomes "54990"

            # --- RATING ---
            # The star rating text is inside a <span class="a-icon-alt">
            # It looks like "4.2 out of 5 stars"
            rating_tag = item.select_one("span.a-icon-alt")
            rating = rating_tag.text.strip() if rating_tag else "N/A"

            # --- IMAGE URL ---
            # The product thumbnail is an <img> with class "s-image"
            # We grab its "src" attribute which holds the image URL
            img_tag = item.select_one("img.s-image")
            image = img_tag["src"] if img_tag else "N/A"

            # --- AD vs ORGANIC ---
            # Sponsored (Ad) products have a label with "Sponsored" text
            # We look for the span that Amazon uses to mark sponsored items
            sponsored_tag = item.select_one("span.s-label-popover-default")
            result_type = "Ad" if sponsored_tag else "Organic"

            # --- STORE THIS PRODUCT ---
            # Create a dictionary with all the fields and add it to our list
            products.append({
                "Title":       title,
                "Price (INR)": price,
                "Rating":      rating,
                "Image URL":   image,
                "Type":        result_type,
            })

        # Pause 2 seconds between pages - this is polite scraping
        # Without this, Amazon detects rapid requests and may block your IP
        time.sleep(2)

    # --- SAVE TO CSV ---
    # Create a timestamp string like "20240519_143022"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"amazon_laptops_{timestamp}.csv"   # e.g. "amazon_laptops_20240519_143022.csv"

    # Convert our list of dicts into a DataFrame (a table)
    df = pd.DataFrame(products)

    # Save the DataFrame to a CSV file
    # index=False means don't write row numbers (0,1,2...) into the file
    df.to_csv(filename, index=False)

    print(f"\nDone! Saved {len(products)} products to '{filename}'")
    return df   # return the DataFrame in case you want to inspect it in a notebook

# --- RUN THE SCRIPT ---
# This block only runs when you execute this file directly
# (not when it's imported by another script)
if __name__ == "__main__":
    df = scrape_amazon_laptops(pages=3)   # scrape 3 pages (~60 products)
    print(df.head(10))                       # show the first 5 rows in the console

Scraping page 1: https://www.amazon.in/s?k=laptops&page=1
  Found 22 products on page 1
Scraping page 2: https://www.amazon.in/s?k=laptops&page=2
  Found 22 products on page 2
Scraping page 3: https://www.amazon.in/s?k=laptops&page=3
  Found 22 products on page 3

Done! Saved 66 products to 'amazon_laptops_20260519_211409.csv'
  Title Price (INR)              Rating  \
0   N/A       70990  5.0 out of 5 stars   
1   N/A       61377  4.1 out of 5 stars   
2   N/A       11990  3.1 out of 5 stars   
3   N/A       35990  3.5 out of 5 stars   
4   N/A      145990                 N/A   
5   N/A       45990  4.0 out of 5 stars   
6   N/A       50990  4.2 out of 5 stars   
7   N/A       48900  4.2 out of 5 stars   
8   N/A       95990                 N/A   
9   N/A       39790  4.0 out of 5 stars   

                                           Image URL     Type  
0  https://m.media-amazon.com/images/I/71odoZVRga...  Organic  
1  https://m.media-amazon.com/images/I/51+50luRJk...  Organic  
2  ht